In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import joblib
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
import unicodedata

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Bidirectional

In [ ]:
# --- Função para criar dados com janela look_back ---
def criar_dados(dataset, look_back=10):
    X, y = [], []
    for i in range(len(dataset) - look_back):
        X.append(dataset[i:i+look_back])
        y.append(dataset[i+look_back])
    return np.array(X), np.array(y)

# --- Função para previsão recursiva ---
def previsao_recursiva(model, input_seq, n_steps):
    """Recebe modelo, sequência inicial (shape=(look_back,1)) e faz previsão para n_steps à frente"""
    input_seq = input_seq.reshape(1, input_seq.shape[0], 1)
    preds = []
    seq = input_seq.copy()

    for _ in range(n_steps):
        pred = model.predict(seq)[0,0]
        preds.append(pred)
        # Atualiza sequência: remove o primeiro, adiciona a previsão no final
        seq = np.append(seq[:,1:,:], [[[pred]]], axis=1)
    return np.array(preds)

# --- Preparação dos dados ---
path_csv = os.path.join('dados_petroleo.csv')
df = pd.read_csv(path_csv, sep=',')
df.columns = [
    unicodedata.normalize('NFKD', col).encode('ASCII', 'ignore').decode('ASCII').strip()
    for col in df.columns.str.replace('"', '')
]
df['Ultimo'] = (
    df['Ultimo'].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False).astype(float)
              )

precos = df['Ultimo'].values.reshape(-1, 1)
scaler = MinMaxScaler(feature_range=(0, 1))
precos_norm = scaler.fit_transform(precos)

look_back = 10
X, y = criar_dados(precos_norm, look_back)
X = X.reshape((X.shape[0], X.shape[1], 1))

tamanho_treino = int(len(X) * 0.8)
X_train, X_test = X[:tamanho_treino], X[tamanho_treino:]
y_train, y_test = y[:tamanho_treino], y[tamanho_treino:]

# --- Modelo LSTM original ---
model_lstm = Sequential([
    LSTM(50, input_shape=(look_back, 1)),
    Dense(1)
])
model_lstm.compile(optimizer='adam', loss='mean_squared_error')
model_lstm.fit(X_train, y_train, epochs=60, batch_size=16, validation_data=(X_test, y_test), verbose=0)

# --- Modelo Bi-LSTM ---
model_bilstm = Sequential([
    Bidirectional(LSTM(50), input_shape=(look_back, 1)),
    Dense(1)
])
model_bilstm.compile(optimizer='adam', loss='mean_squared_error')
model_bilstm.fit(X_train, y_train, epochs=60, batch_size=16, validation_data=(X_test, y_test), verbose=0)

# --- Previsão no conjunto de teste para avaliação ---
y_pred_lstm = model_lstm.predict(X_test)
y_pred_bilstm = model_bilstm.predict(X_test)

y_test_inv = scaler.inverse_transform(y_test.reshape(-1, 1))
y_pred_lstm_inv = scaler.inverse_transform(y_pred_lstm)
y_pred_bilstm_inv = scaler.inverse_transform(y_pred_bilstm)

# MSE dos modelos (avaliando só no teste)
mse_lstm = mean_squared_error(y_test_inv, y_pred_lstm_inv)
mse_bilstm = mean_squared_error(y_test_inv, y_pred_bilstm_inv)
print(f'MSE LSTM: {mse_lstm:.2f}')
print(f'MSE Bi-LSTM: {mse_bilstm:.2f}')

# --- Previsão recursiva de 30 dias ---
n_dias = 30
ultimo_input = precos_norm[-look_back:].reshape(-1, 1)

preds_lstm_norm = previsao_recursiva(model_lstm, ultimo_input, n_dias)
preds_bilstm_norm = previsao_recursiva(model_bilstm, ultimo_input, n_dias)

# Inverter escala para valores reais
preds_lstm = scaler.inverse_transform(preds_lstm_norm.reshape(-1, 1))
preds_bilstm = scaler.inverse_transform(preds_bilstm_norm.reshape(-1, 1))

# --- Gráfico comparativo ---
plt.figure(figsize=(12,6))
plt.plot(range(1, n_dias+1), preds_lstm, label='Previsão LSTM')
plt.plot(range(1, n_dias+1), preds_bilstm, label='Previsão Bi-LSTM')
plt.title('Previsão do preço do petróleo para os próximos 30 dias')
plt.xlabel('Dias futuros')
plt.ylabel('Preço do barril')
plt.legend()
plt.grid(True)
plt.show()